In [1]:
## Document structure

from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content = 'Machine Learning (ML) is not magic. It’s applied statistics at scale, wrapped in code, fueled by data. If you treat it like buzzword decoration, you’ll build fragile systems. If you treat it like disciplined engineering, you’ll build leverage.',
    metadata = {
        'source' : 'example.txt',
        'page' : 1,
        'author' : 'self',
        'date_created' : '2026-01-01'
    }
)

doc

Document(metadata={'source': 'example.txt', 'page': 1, 'author': 'self', 'date_created': '2026-01-01'}, page_content='Machine Learning (ML) is not magic. It’s applied statistics at scale, wrapped in code, fueled by data. If you treat it like buzzword decoration, you’ll build fragile systems. If you treat it like disciplined engineering, you’ll build leverage.')

In [3]:
## Create a simple txt flie

import os 
os.makedirs('../DATA/text_files',exist_ok = True)

In [4]:
sample_texts = {
    "../DATA/text_files/RAG.txt":"""
    
“Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks” is a 2020 Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks paper by Patrick Lewis and colleagues that introduced the now-standard “RAG” framework. It shows how to combine neural retrieval over a large corpus with sequence-to-sequence generation to tackle tasks where factual knowledge and up-to-date information are crucial.

Key facts
Authors: Patrick Lewis, Ethan Perez, Aleksandra Piktus, Fabio Petroni, et al.

Published at: NeurIPS 2020 (with earlier arXiv preprint).

Core idea: Combine parametric seq2seq model with non-parametric memory (Wikipedia index).

Retriever: Dense Passage Retrieval (DPR) over Wikipedia.

Impact: Established “RAG” as a standard approach for grounded, knowledge-intensive NLP.

Background and motivation
The paper starts from the observation that large pre-trained language models (LMs) implicitly store factual knowledge in their parameters, but are hard to update and often struggle on knowledge-intensive tasks like open-domain QA, fact verification, and entity-centric reasoning. They also provide poor provenance: it’s unclear where a fact came from. RAG is proposed to fix these issues by letting a generator query an explicit external knowledge source at inference time.

Model architecture
RAG combines:

Parametric memory: a pre-trained seq2seq model (like BART) that handles language understanding and generation.

Non-parametric memory: a dense vector index of Wikipedia passages, built using Dense Passage Retrieval (DPR).

Given a query, DPR retrieves top-k relevant passages. RAG then marginalizes over these passages while generating an answer, effectively treating each passage as a latent variable. The paper proposes two variants:

RAG-Sequence: conditions on the same retrieved set for the whole sequence.

RAG-Token: allows different tokens to attend to different passages, giving more flexibility.

Training and tasks
The authors present a general fine-tuning recipe: jointly train the retriever and generator (initializing both from strong pre-trained models) on downstream supervision. They evaluate on multiple knowledge-intensive benchmarks, including open-domain QA datasets like Natural Questions, TriviaQA, and WebQuestions, and broader tasks such as slot filling and dialog-style QA.

Across tasks, RAG models set new state of the art at the time, outperforming:

Purely parametric seq2seq models (no retrieval).

Traditional retrieve-and-extract pipelines that generate answers by span extraction rather than free-form generation.

Notable findings
Factuality and specificity: RAG generations are more specific, diverse, and factually accurate compared with a strong parametric-only baseline, thanks to access to explicit passages.

Provenance: Because outputs are conditioned on retrieved documents, systems can surface those documents as evidence, improving transparency.

Updateability: Updating knowledge no longer requires retraining the generator; updating or swapping the document index is enough, which is crucial for dynamic domains.

Impact and legacy
This paper effectively coined “retrieval-augmented generation” and catalyzed a large line of work on RAG-style systems. Later research has extended it with better retrievers, multi-hop reasoning, evidentiality modeling, and complex LM–retriever pipelines, but Lewis et al. (2020) remains the canonical reference for the original RAG framework used in many modern knowledge-grounded LLM applications.




    
    """
}

for filepath, content in sample_texts.items():
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)

print('file created')

file created


In [5]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader('../DATA/text_files/RAG.txt', encoding = 'utf-8')
document = loader.load()

print(document)

d:\Data Science Project\RAG-Pipeline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../DATA/text_files/RAG.txt'}, page_content='\n\n“Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks” is a 2020 Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks paper by Patrick Lewis and colleagues that introduced the now-standard “RAG” framework. It shows how to combine neural retrieval over a large corpus with sequence-to-sequence generation to tackle tasks where factual knowledge and up-to-date information are crucial.\n\nKey facts\nAuthors: Patrick Lewis, Ethan Perez, Aleksandra Piktus, Fabio Petroni, et al.\n\nPublished at: NeurIPS 2020 (with earlier arXiv preprint).\n\nCore idea: Combine parametric seq2seq model with non-parametric memory (Wikipedia index).\n\nRetriever: Dense Passage Retrieval (DPR) over Wikipedia.\n\nImpact: Established “RAG” as a standard approach for grounded, knowledge-intensive NLP.\n\nBackground and motivation\nThe paper starts from the observation that large pre-trained language models (LMs) imp

In [6]:
### Directory Loader

from langchain_community.document_loaders import DirectoryLoader

## Load all the text files from the directory

dir_loader = DirectoryLoader(
    "../DATA/text_files",
    glob = "**/*.txt", ## Pattern to match files
    loader_cls = TextLoader, ## Loader class to use
    loader_kwargs = {'encoding':'utf-8'},
    show_progress = False
)

documents = dir_loader.load()
documents 

[Document(metadata={'source': '..\\DATA\\text_files\\ML.txt'}, page_content='Machine Learning (ML) is not magic. It’s applied statistics at scale, wrapped in code, fueled by data. If you treat it like buzzword decoration, you’ll build fragile systems. If you treat it like disciplined engineering, you’ll build leverage.\n\nAt its core, ML is about designing systems that learn patterns from data and improve performance without being explicitly programmed for every rule. Instead of hard-coding logic (“if X, then Y”), you feed data into algorithms that infer structure and make predictions.\n\nThe Three Core Paradigms\n\nSupervised Learning\n\nLearn from labeled data.\n\nTasks: classification (spam detection), regression (house price prediction).\n\nAlgorithms: Linear Regression, Logistic Regression, Decision Trees, Neural Networks.\n\nUnsupervised Learning\n\nNo labels. Discover hidden structure.\n\nTasks: clustering, dimensionality reduction.\n\nAlgorithms: K-Means, Hierarchical Clusterin

In [21]:
## Web based loader

from langchain_community.document_loaders import WebBaseLoader
import bs4
web_loader = WebBaseLoader(web_path = "https://transformer-circuits.pub/2021/framework/index.html" ,
                            bs_kwargs = dict(parse_only = bs4.SoupStrainer(
                                class_ = ("1-body", "d-byline-container base-grid")
                            ))

)
doc = web_loader.load()
doc

[Document(metadata={'source': 'https://transformer-circuits.pub/2021/framework/index.html'}, page_content='   Authors Nelson Elhage∗†, Neel Nanda∗, Catherine Olsson∗, Tom Henighan†, Nicholas Joseph†, Ben Mann†, Amanda Askell, Yuntao Bai, Anna Chen, Tom Conerly, Nova DasSarma, Dawn Drain, Deep Ganguli, Zac Hatfield-Dodds, Danny Hernandez, Andy Jones, Jackson Kernion, Liane Lovitt, Kamal Ndousse, Dario Amodei, Tom Brown, Jack Clark, Jared Kaplan, Sam McCandlish, Chris Olah‡    Affiliation Anthropic   Published Dec 22, 2021    * Core Research Contributor; † Core Infrastructure Contributor; ‡ Correspondence to colah@anthropic.com; Author contributions statement below.    ')]